In [1]:
import os

print(os.listdir('/kaggle/input/datasets'))

['sukanyabasuiem']


In [2]:
print(os.listdir('/kaggle/input/datasets/sukanyabasuiem'))

['midi-dataset']


In [3]:
!pip install music21 tensorflow numpy

In [4]:
from music21 import converter, instrument, note, chord
import glob
import numpy as np

In [5]:
import glob

files = glob.glob(
    '/kaggle/input/datasets/sukanyabasuiem/midi-dataset/**/*.mid',
    recursive=True
)

print("Total MIDI files found:", len(files))
print(files[:5])

Total MIDI files found: 51
['/kaggle/input/datasets/sukanyabasuiem/midi-dataset/x (43).mid', '/kaggle/input/datasets/sukanyabasuiem/midi-dataset/midi_dataset/midi_dataset/x (26).mid', '/kaggle/input/datasets/sukanyabasuiem/midi-dataset/midi_dataset/midi_dataset/x (30).mid', '/kaggle/input/datasets/sukanyabasuiem/midi-dataset/midi_dataset/midi_dataset/x (50).mid', '/kaggle/input/datasets/sukanyabasuiem/midi-dataset/midi_dataset/midi_dataset/x (12).mid']


In [6]:
notes = []

for file in files:
    try:
        midi = converter.parse(file)

        parts = instrument.partitionByInstrument(midi)

        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    except:
        continue

print("Total notes extracted:", len(notes))

/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=7, data=b'The Chainsmokers feat. XYL\xd8 - Setting Fires'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=8, data=b'Shawn Mendes, Camila Cabello \x96 Se\xf1orita'>; getting generic Instrument
  warnings.warn(


Total notes extracted: 3892


In [7]:
# Get all unique notes
pitchnames = sorted(set(notes))

# Create mapping: note → number
note_to_int = {note: number for number, note in enumerate(pitchnames)}

print("Total unique notes:", len(pitchnames))

Total unique notes: 113


In [8]:
import numpy as np

sequence_length = 100

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    seq_in = notes[i:i+sequence_length]
    seq_out = notes[i+sequence_length]

    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

n_patterns = len(network_input)

print("Total patterns:", n_patterns)

Total patterns: 3792


In [9]:
network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))

network_input = network_input / float(len(pitchnames))

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense

model = Sequential()

model.add(LSTM(256, input_shape=(network_input.shape[1], network_input.shape[2]), return_sequences=True))
model.add(Dropout(0.3))

model.add(LSTM(256))
model.add(Dense(128, activation='relu'))

model.add(Dense(len(pitchnames), activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

model.summary()

2026-05-01 00:04:31.273723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777593871.538423      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777593871.611801      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777593872.287853      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777593872.287913      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777593872.287919      57 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 256)       │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 113)            │        14,577 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 836,977 (3.19 MB)

 Trainable params: 836,977 (3.19 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import numpy as np

# Converting input to NumPy array
network_input = np.array(network_input, dtype=np.float32)

# Convert output to NumPy array
network_output = np.array(network_output, dtype=np.int32)

print(type(network_input))
print(network_input.shape)

<class 'numpy.ndarray'>
(3792, 100, 1)


In [12]:
model.fit(
    network_input,
    network_output,
    epochs=20,
    batch_size=64
)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 43s 646ms/step - loss: 4.4653
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 39s 641ms/step - loss: 4.1105
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 634ms/step - loss: 3.9384
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 628ms/step - loss: 3.8559
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 625ms/step - loss: 3.7046
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 630ms/step - loss: 3.5709
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 38s 635ms/step - loss: 3.6287
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 42s 694ms/step - loss: 3.2950
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 45s 756ms/step - loss: 3.1947
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 42s 692ms/step - loss: 2.9776
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 40s 660ms/step - loss: 2.7508
Epoch 12/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 44s 738ms/step - loss: 3.0065
Epoch 13/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 40s 663ms/step - loss: 2.5526
Epoch 14/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 40s 661ms/step - loss: 3.6959
Epoch 15/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 42s 

In [13]:
model.save('/kaggle/working/music_model.keras')
print("Model saved successfully")

Model saved successfully


In [20]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/working/music_model.keras')

In [21]:
def sample_with_temperature(preds, temperature=0.8):
    preds = np.asarray(preds).astype('float64')

    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)

    return np.random.choice(len(preds), p=preds)

import random

# Picking a random starting point from my training data
start = random.randint(0, len(network_input)-1)

# Getting initial pattern (sequence of 100 notes)
pattern = network_input[start]

prediction_output = []  # This will store generated notes

for i in range(200):  # Generating 200 notes

    # Reshaping input to match model input format
    prediction_input = np.reshape(pattern, (1, len(pattern), 1))

    # Predicting next note probabilities
    prediction = model.predict(prediction_input, verbose=0)

    # Getting index with highest probability
    prediction = prediction[0]  # because model returns batch output
    index = sample_with_temperature(prediction, temperature=0.8)

    # Converting index back to note
    result = pitchnames[index]
    prediction_output.append(result)

    # Adding predicted note to pattern and remove first (sliding window)
    pattern = np.append(pattern, index)
    pattern = pattern[1:]

In [22]:
from music21 import stream, note, chord

output_notes = []

for pattern in prediction_output:

    if ('.' in pattern) or pattern.isdigit():
        # If it's a chord
        notes_in_chord = pattern.split('.')
        notes = [note.Note(int(n)) for n in notes_in_chord]
        new_chord = chord.Chord(notes)
        output_notes.append(new_chord)

    else:
        # If it's a single note
        new_note = note.Note(pattern)
        output_notes.append(new_note)

# Creating music stream
midi_stream = stream.Stream(output_notes)

# Save as MIDI file
midi_stream.write('midi', fp='output.mid')

'output.mid'